# Initial Model Baselines

## Objective

Compare a no-failure baseline and logistic regression using the fixed
train/validation split.

Both pipelines are trained on training data only. Initial decisions use
a probability threshold of 0.5. Test data remains reserved.

Business costs are hypothetical:
- Missed failure: €10,000.
- Any maintenance intervention: €1,000.

An intervention on a failure case is assumed to avoid the failure cost.

In [1]:
from pathlib import Path

import pandas as pd

from predictive_maintenance.data import split_dataset, validate_raw_data
from predictive_maintenance.models import build_baseline_pipelines
from predictive_maintenance.preprocessing import FEATURE_COLUMNS

df = pd.read_csv(
    Path("../data/raw/ai4i2020.csv"),
    encoding="utf-8-sig",
)

validate_raw_data(df)

train_df, val_df, test_df = split_dataset(df, random_state=42)

X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df["Machine failure"].copy()

X_val = val_df[FEATURE_COLUMNS].copy()
y_val = val_df["Machine failure"].copy()

## 1. Train baseline pipelines

Fit a separate preprocessing-and-model pipeline for each baseline.
No validation or test observations are used during fitting.

In [2]:
pipelines = build_baseline_pipelines()

for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    print(f"{name}: training completed")

dummy: training completed
logistic_regression: training completed


## 2. Validation performance at threshold 0.5

Precision, recall, F1, confusion counts, and business cost are computed
from thresholded decisions.

ROC-AUC and Average Precision are computed from continuous scores.
Average Precision is reported explicitly; it is not the trapezoidal
area under the precision-recall curve.

The simulated cost includes interventions for both true and false positives.

In [3]:
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

THRESHOLD = 0.5
COST_MISSED_FAILURE = 10_000
COST_INTERVENTION = 1_000

results = []

for name, pipeline in pipelines.items():
    positive_index = list(pipeline.classes_).index(1)
    probabilities = pipeline.predict_proba(X_val)[:, positive_index]
    predictions = (probabilities >= THRESHOLD).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    business_cost = (
        COST_MISSED_FAILURE * fn
        + COST_INTERVENTION * (tp + fp)
    )

    results.append({
        "model": name,
        "precision": precision_score(y_val, predictions, zero_division=0),
        "recall": recall_score(y_val, predictions, zero_division=0),
        "f1": f1_score(y_val, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_val, probabilities),
        "average_precision": average_precision_score(y_val, probabilities),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "business_cost_eur": int(business_cost),
    })

results_df = pd.DataFrame(results).set_index("model")
results_df.round(4).T

model,dummy,logistic_regression
precision,0.000,0.7647
recall,0.000,0.1912
f1,0.000,0.3059
roc_auc,0.500,0.8792
average_precision,0.034,0.4054
tn,1932.000,1928.0000
fp,0.000,4.0000
fn,68.000,55.0000
tp,0.000,13.0000
business_cost_eur,680000.000,567000.0000


## 3. Initial validation findings

At threshold 0.5, logistic regression detects 13 of 68 failures and
produces 4 false alarms.

- Precision: 76.47%.
- Recall: 19.12%.
- ROC-AUC: 0.8792.
- Average Precision: 0.4054.

The model ranks observations better than the constant baseline,
but misses most failures at the current threshold.

Simulated validation cost decreases from €680,000 to €567,000,
a 16.62% reduction relative to the no-intervention baseline.

This improvement comes from the model at threshold 0.5, not from
threshold optimization. Costs are hypothetical, and these are
validation results, not final test results.

Next, we will investigate the decision threshold using validation data.